In [1]:
import geopandas as gpd
import pandas as pd
import polars as pl

In [2]:
demanda_scan = pl.scan_parquet(r"../outputs/01/Dados_4Meses_2023_2024.parquet")
demanda_lazy = demanda_scan.select(["data", "linha_blt", "zona_emb"])  # Pega só o que precisa

# Registros sem linha_blt nao sao embarques de onibus municipal (ver diagnostico em
# 04_diagnostico_domingos_antiga_vs_nova.ipynb) - excluidos de toda contagem de demanda.
demanda_lazy = demanda_lazy.filter(pl.col("linha_blt").is_not_null())

In [3]:
demanda_lazy = demanda_lazy.with_columns(
    pl.col("data").str.slice(0, 4).alias("Ano"),
    pl.col("data").str.slice(4, 2).alias("Mês")
)

In [4]:
demanda_mes = (
    demanda_lazy
    .group_by(["linha_blt", "zona_emb", "Ano", "Mês"])
    .agg(
        pl.len().alias("N_embarques")
    )
    .collect(engine="streaming")  # roda tudo (leitura + filtro + agregação) em streaming, sem materializar a base inteira
)

In [5]:
demanda_mes

linha_blt,zona_emb,Ano,Mês,N_embarques
cat,f64,str,str,u32
"""648P-10""",313.0,"""2023""","""04""",43022
"""5010-10""",280.0,"""2023""","""04""",2043
"""675K-10""",54.0,"""2023""","""04""",90782
"""6970-10""",288.0,"""2023""","""04""",4233
"""8610-10""",333.0,"""2023""","""04""",32575
…,…,…,…,…
"""2731-10""",198.0,"""2024""","""04""",24
"""2100-10""",17.0,"""2024""","""04""",3
"""737A-10""",68.0,"""2024""","""05""",2


In [6]:
demanda_mes.write_parquet("../outputs/01/Dados_4Meses_2023_2024_mes.parquet")